# Demo: Ingestion de Documentos

Este notebook demuestra cómo procesar documentos y construir un grafo de conocimiento.

In [1]:
import sys
from dotenv import load_dotenv
sys.path.append('..')
load_dotenv("../env/.env")

from graphrag.graph.neo4j_manager import Neo4jManager
from graphrag.ingestion.text_processor import TextProcessor

/home/Pablo/Universidad/02-segundo-cuatrimestre/IAC/GraphRAG-IAC/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Inicializar Neo4j

In [2]:
neo4j = Neo4jManager()
schema = neo4j.get_schema()
schema = neo4j.format_schema(schema)
print(schema)
# neo4j.create_constraints()
# neo4j.create_vector_index()

Node labels and properties:
:`Document` {id: STRING NOT NULL, title: STRING NOT NULL, source: STRING NOT NULL}
:`Species` {name: STRING NOT NULL, length_max_m: FLOAT NOT NULL, height_max_m: FLOAT NOT NULL, weight_max_kg: FLOAT NOT NULL, maturity_age_years: FLOAT NOT NULL, gestation_period_days: FLOAT NOT NULL, lifespan_years: FLOAT NOT NULL, weight_min_kg: FLOAT NOT NULL, top_speed_kmh: FLOAT NOT NULL, length_min_m: FLOAT NOT NULL, height_min_m: FLOAT NOT NULL}
:`Family` {type: STRING NOT NULL}
:`AnimalClass` {type: STRING NOT NULL}
:`SkeletalStructure` {type: STRING NOT NULL}
:`ReproductionMethod` {type: STRING NOT NULL}
:`EnvironmentType` {type: STRING NOT NULL}
:`Habitat` {type: STRING NOT NULL}
:`Location` {type: STRING NOT NULL}
:`ActivityCycle` {type: STRING NOT NULL}
:`SocialStructure` {type: STRING NOT NULL}
:`DietType` {type: STRING NOT NULL}
:`FoodSource` {type: STRING NOT NULL}
:`ConservationStatus` {type: STRING NOT NULL}
:`Question` {text: STRING NOT NULL, question_embeddi

In [3]:
animals = [
    "Tiger", "Lion", "Elephant", "Dolphin", "Giant Panda", "Horse", "Penguin", "Wolf", "Shark", "Rabbit",
    "Gorilla", "Giraffe", "Cheetah", "Polar Bear", "Hippopotamus", "Zebra", "Red Panda", "Kangaroo", "Koala", "Sloth",
    "Meerkat", "Rhinoceros", "Snow Leopard", "Orangutan", "Chimpanzee", "Platypus", "Lemur", "Capybara", "Sea Otter", "Arctic Fox",
    "Jaguar", "Black Panther", "Brown Bear", "Okapi", "Tasmanian Devil", "Wombat", "Quokka", "Hedgehog", "Fennec Fox", "Bald Eagle",
    "Peacock", "Flamingo", "Parrot", "Hummingbird", "Swan", "Toucan", "Owl", "Puffin", "Woodpecker", "Raven",
    "Falcon", "Ostrich", "Emu", "Albatross", "Robin", "Blue Jay", "Cardinal", "Canary", "Guinea Pig", "Hamster",
    "Cow", "Pig", "Sheep", "Goat", "Chicken", "Duck", "Donkey", "Llama", "Alpaca", "Ferret",
    "Chinchilla", "Great White Shark", "Blue Whale", "Killer Whale (Orca)", "Sea Turtle", "Octopus", "Seahorse", "Clownfish", "Axolotl", "Manatee",
    "Stingray", "Walrus", "Seal", "Narwhal", "Lobster", "Jellyfish", "Chameleon", "Komodo Dragon", "King Cobra", "Bearded Dragon",
    "Green Iguana", "Tree Frog", "Monarch Butterfly", "Honey Bee", "Praying Mantis", "Ladybug", "Raccoon", "Camel", "Squirrel", "Buffalo"
]

print(f"Total animals: {len(animals)}")
print(animals)

Total animals: 100
['Tiger', 'Lion', 'Elephant', 'Dolphin', 'Giant Panda', 'Horse', 'Penguin', 'Wolf', 'Shark', 'Rabbit', 'Gorilla', 'Giraffe', 'Cheetah', 'Polar Bear', 'Hippopotamus', 'Zebra', 'Red Panda', 'Kangaroo', 'Koala', 'Sloth', 'Meerkat', 'Rhinoceros', 'Snow Leopard', 'Orangutan', 'Chimpanzee', 'Platypus', 'Lemur', 'Capybara', 'Sea Otter', 'Arctic Fox', 'Jaguar', 'Black Panther', 'Brown Bear', 'Okapi', 'Tasmanian Devil', 'Wombat', 'Quokka', 'Hedgehog', 'Fennec Fox', 'Bald Eagle', 'Peacock', 'Flamingo', 'Parrot', 'Hummingbird', 'Swan', 'Toucan', 'Owl', 'Puffin', 'Woodpecker', 'Raven', 'Falcon', 'Ostrich', 'Emu', 'Albatross', 'Robin', 'Blue Jay', 'Cardinal', 'Canary', 'Guinea Pig', 'Hamster', 'Cow', 'Pig', 'Sheep', 'Goat', 'Chicken', 'Duck', 'Donkey', 'Llama', 'Alpaca', 'Ferret', 'Chinchilla', 'Great White Shark', 'Blue Whale', 'Killer Whale (Orca)', 'Sea Turtle', 'Octopus', 'Seahorse', 'Clownfish', 'Axolotl', 'Manatee', 'Stingray', 'Walrus', 'Seal', 'Narwhal', 'Lobster', 'Jelly

## 2. Cargar documento de ejemplo

In [4]:
FILE_PATH = "../scraping/data/lion.md"
with open(FILE_PATH, 'r', encoding='utf-8') as f:
    markdown_text = f.read()

print(markdown_text)

# Lion

## Lion Facts

#### The lion is Africa’s apex predator

The lion is one of the largest, strongest, and most powerful felines in the world, second only in size to the Siberian Tiger. They are the largest cats on the African continent.

While most big cats are solitary hunters, lions are incredibly sociable animals that live together in family groups called pride.

They are some of the world’s most popular animals :

## Scientific Name and Classification

The scientific name for lions is Panthera leo . The genus Panthera is of Greek origin and comprises big cat species such as tigers , lions, jaguars , and leopards that have the ability to roar. Leo is the Latin word for lion.

There are two types of lion subspecies. One is named Panthera leo melanochaita and lives across South and East Africa. The second lion subspecies has the scientific name Panther Leo and lives in West Africa, Central Africa, and Asia.

You may see references to African and Asiatic lions. Up until 2017, ther

## 3. Procesar documento

In [5]:
processor = TextProcessor(neo4j_manager=neo4j, species_names=animals, chunk_size=800, chunk_overlap=80)

processor.process_document(
    text=markdown_text,
    document_id="lion",
    metadata={"title": "Lion", "source": "A-Z Animals"}
)

Documento dividido en 19 chunks


Procesando chunks:   0%|          | 0/19 [00:00<?, ?it/s]

Initial extraction: 2 entities and 1 relationships
Audit completed:
 - Entities: Deleted 1 | Added 0 -> Final: 1
 - Relationships: Deleted 1 | Added 2 -> Final: 2


Procesando chunks:   5%|▌         | 1/19 [00:35<10:34, 35.22s/it]

Initial extraction: 13 entities and 16 relationships
Audit completed:
 - Entities: Deleted 7 | Added 1 -> Final: 7
 - Relationships: Deleted 12 | Added 0 -> Final: 4
Especie today’s surviving lions descartada


...

Initial extraction: 1 entities and 3 relationships
Audit completed:
 - Entities: Deleted 2 | Added 0 -> Final: 1
 - Relationships: Deleted 2 | Added 0 -> Final: 1
Especie herbivores descartada


Procesando chunks:  95%|█████████▍| 18/19 [09:16<00:26, 26.91s/it]

Initial extraction: 2 entities and 1 relationships
Audit completed:
 - Entities: Deleted 1 | Added 1 -> Final: 2
 - Relationships: Deleted 1 | Added 1 -> Final: 1


Procesando chunks: 100%|██████████| 19/19 [09:40<00:00, 30.53s/it]

Deleted 1 self-loops of the relationship 'PREYS_ON'
Documento lion procesado exitosamente


## 4. Verificar el grafo

In [6]:
# Contar nodos
stats = neo4j.execute_query("""
MATCH (n)
RETURN labels(n)[0] as label, count(*) as count
ORDER BY count DESC
""")

for stat in stats:
    print(f"{stat['label']}: {stat['count']}")

Question: 206
Chunk: 33
Location: 17
Species: 12
Habitat: 8
ConservationStatus: 5
SocialStructure: 1
Document: 1
Family: 1
EnvironmentType: 1
DietType: 1
AnimalClass: 1
ActivityCycle: 1


In [7]:
# Ver algunas entidades
entities = neo4j.execute_query("""
MATCH (e:Entity)
RETURN e.name as name
LIMIT 10
""")

for entity in entities:
    print(f"\n{entity['name']} ({entity['type']})")
    print(f"  {entity['summary'][:100]}..." if entity['summary'] else "  No summary")

Received notification from DBMS server: <GqlStatusObject gql_status='01N50', status_description='warn: label does not exist. The label `Entity` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=2, column=10, offset=10>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 10, 'line': 2, 'column': 10}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: '\nMATCH (e:Entity)\nRETURN e.name as name\nLIMIT 10\n'


In [ ]:
# Ver algunas relaciones
relationships = neo4j.execute_query("""
MATCH (s:Entity)-[r:RELATIONSHIP]->(t:Entity)
RETURN s.name as source, r.type as type, t.name as target, r.summary as summary
LIMIT 10
""")

for rel in relationships:
    print(f"\n{rel['source']} -[{rel['type']}]-> {rel['target']}")
    print(f"  {rel['summary'][:100]}..." if rel['summary'] else "  No summary")

## 5. Cleanup (opcional)

In [ ]:
# Descomentar para limpiar la base de datos
# neo4j.execute_query("MATCH (n) DETACH DELETE n")
neo4j.close()